In [1]:
# ========================================================================
# EXPERIMENT 1: CONTROL (Original Images)
# FIX: Using EfficientNetV2B2 and IMAGE_SIZE=260 to fit in memory
# ========================================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, GlobalAveragePooling2D, Dropout, Dense)
# --- FIX: Using a slightly smaller, but still very powerful, model ---
from tensorflow.keras.applications import EfficientNetV2B2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Enable mixed precision for faster training
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision enabled:", tf.keras.mixed_precision.global_policy().name)

Mixed precision enabled: mixed_float16


In [ ]:
# ========================================================================
# SECTION 1: CONFIGURATION (OPTIMIZED FOR FITZPATRICK CLASSIFICATION)
# ========================================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.applications import EfficientNetV2B2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# ========================================================================
# KEY CONFIGURATION: TOGGLE BETWEEN 3-WAY AND 6-WAY CLASSIFICATION
# ========================================================================
USE_3WAY_CLASSIFICATION = True  # Set to False for 6-way classification

# Paths
BASE_PATH = r'D:\skin cancer project\datasets\MSKCC-images'
CSV_PATH = os.path.join(r'D:\skin cancer project\datasets', 'mskcc-skin-tone-labeling-dataset_metadata_2025-11-24.csv')

# Image settings
IMAGE_SIZE = 260
BATCH_SIZE = 8  # Reduced for small dataset

# Training epochs (increased for small dataset)
FEATURE_EXTRACTION_EPOCHS = 15
FINE_TUNING_EPOCHS = 60

# Learning rates
LEARNING_RATE_HEAD = 1e-3
LEARNING_RATE_FINE_TUNE_INITIAL = 5e-6  # Reduced for stability

# Regularization
DROPOUT_RATE = 0.5  # Increased from 0.4
LABEL_SMOOTHING = 0.1

# Focal loss parameters (for handling imbalance)
FOCAL_LOSS_GAMMA = 2.0
FOCAL_LOSS_ALPHA = 0.25

AUTOTUNE = tf.data.AUTOTUNE

# Enable mixed precision
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision enabled:", tf.keras.mixed_precision.global_policy().name)

print(f"\n{'='*60}")
print(f"CONFIGURATION: {'3-WAY' if USE_3WAY_CLASSIFICATION else '6-WAY'} FITZPATRICK CLASSIFICATION")
print(f"{'='*60}")


In [ ]:
"""
if
USE_3WAY_CLASSIFICATION = False

BATCH_SIZE = 16  # or 32
FEATURE_EXTRACTION_EPOCHS = 10
FINE_TUNING_EPOCHS = 40
DROPOUT_RATE = 0.4

else
USE_3WAY_CLASSIFICATION = True

BATCH_SIZE = 8  # Reduced for small dataset
FEATURE_EXTRACTION_EPOCHS = 15
FINE_TUNING_EPOCHS = 60
DROPOUT_RATE = 0.5

"""

In [ ]:
# ========================================================================
# SECTION 2: DATA LOADING WITH FITZPATRICK CLASSIFICATION
# ========================================================================

print("\n--- Loading and preparing data (FITZPATRICK SKIN TYPE CLASSIFICATION) ---")
df = pd.read_csv(CSV_PATH)
df['path'] = df['isic_id'].apply(lambda x: os.path.join(BASE_PATH, x + '.jpg'))

# Remove rows with missing Fitzpatrick labels
df = df.dropna(subset=['fitzpatrick_skin_type', 'path'])
print(f"Total images after removing missing labels: {len(df)}")

# Class grouping function
def group_fitzpatrick(skin_type):
    """Group Fitzpatrick types into 3 broader categories"""
    if skin_type in ['I', 'II']:
        return 'Light'
    elif skin_type in ['III', 'IV']:
        return 'Medium'
    else:  # V, VI
        return 'Dark'

# Apply grouping or use original labels based on configuration
if USE_3WAY_CLASSIFICATION:
    df['target_label'] = df['fitzpatrick_skin_type'].apply(group_fitzpatrick)
    print("\n✓ Using 3-way classification: Light / Medium / Dark")
else:
    df['target_label'] = df['fitzpatrick_skin_type']
    print("\n✓ Using 6-way classification: I / II / III / IV / V / VI")

# Show class distribution
print("\nClass distribution:")
class_dist = df.groupby('target_label').agg({
    'patient_id': 'nunique',
    'isic_id': 'count'
}).rename(columns={'patient_id': 'Patients', 'isic_id': 'Images'})
print(class_dist)

# Create one-hot encoded labels
label_columns = sorted(df['target_label'].unique())
df = pd.concat([df, pd.get_dummies(df['target_label'], dtype='float32')], axis=1)

print(f"\nNumber of classes: {len(label_columns)}")
print(f"Classes: {label_columns}")

# ========================================================================
# PATIENT-BASED SPLIT (80/10/10)
# ========================================================================

print("\n--- Splitting by Patient ID (80/10/10) ---")
unique_patients = df['patient_id'].unique()
n_patients = len(unique_patients)
print(f"Total patients: {n_patients}")

# First split: 80% train, 20% temp (for val+test)
train_patients, temp_patients = train_test_split(
    unique_patients, 
    test_size=0.2,
    random_state=42
)

# Second split: Split the 20% into 10% validation and 10% test
val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.5,
    random_state=42
)

# Filter dataframe by patient assignments
train_df = df[df['patient_id'].isin(train_patients)].copy()
validation_df = df[df['patient_id'].isin(val_patients)].copy()
test_df = df[df['patient_id'].isin(test_patients)].copy()

# Verification: Ensure no patient overlap
assert len(set(train_patients) & set(val_patients)) == 0, "ERROR: Patient overlap between train and validation!"
assert len(set(train_patients) & set(test_patients)) == 0, "ERROR: Patient overlap between train and test!"
assert len(set(val_patients) & set(test_patients)) == 0, "ERROR: Patient overlap between validation and test!"

print(f"\n✓ Patient-based split complete (no overlap verified):")
print(f"  Training:   {len(train_patients):2d} patients ({len(train_patients)/n_patients*100:.1f}%) → {len(train_df):4d} images")
print(f"  Validation: {len(val_patients):2d} patients ({len(val_patients)/n_patients*100:.1f}%) → {len(validation_df):4d} images")
print(f"  Test:       {len(test_patients):2d} patients ({len(test_patients)/n_patients*100:.1f}%) → {len(test_df):4d} images")

# ========================================================================
# COMPUTE CLASS WEIGHTS FOR IMBALANCED DATA
# ========================================================================

print("\n--- Computing class weights for imbalanced data ---")

# Get all training labels
train_labels = train_df['target_label'].values

# Compute class weights
unique_classes = np.unique(train_labels)
class_weights_array = compute_class_weight(
    'balanced',
    classes=unique_classes,
    y=train_labels
)

# Create dictionary mapping class index to weight
# Need to map string labels to indices matching one-hot encoding
label_to_index = {label: idx for idx, label in enumerate(label_columns)}
class_weight_dict = {label_to_index[cls]: weight for cls, weight in zip(unique_classes, class_weights_array)}

print("\nClass weights:")
for label in label_columns:
    idx = label_to_index[label]
    weight = class_weight_dict.get(idx, 1.0)
    count = (train_labels == label).sum()
    print(f"  {label:10s}: {weight:.3f} (n={count})")



--- Loading and preparing data (CONTROL - ORIGINAL IMAGES) ---

--- Splitting by Patient ID (80/10/10) ---
Total patients: 64

✓ Patient-based split complete (no overlap verified):
  Training:   51 patients (79.7%) → 3824 images
  Validation:  6 patients (9.4%) →  511 images
  Test:        7 patients (10.9%) →  544 images


In [ ]:
# ========================================================================
# SECTION 3: OPTIMIZED DATA PIPELINE (SKIN-TONE FOCUSED AUGMENTATION)
# ========================================================================

print(f"\n--- Building tf.data pipelines (IMAGE_SIZE={IMAGE_SIZE})...")

def parse_image(filepath, label):
    """Load and resize image"""
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], method=tf.image.ResizeMethod.BILINEAR)
    return image, label

def skin_tone_augment(image, label):
    """
    Skin-tone-specific augmentation.
    Focus on COLOR variations (critical for skin classification).
    NO CutMix - it creates unrealistic mixed skin tones!
    """
    # Spatial augmentation
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    
    # Color augmentation (MORE IMPORTANT for skin tone classification)
    image = tf.image.random_hue(image, max_delta=0.05)  # Subtle hue variations
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)  # Saturation changes
    image = tf.image.random_brightness(image, max_delta=0.15)  # Lighting conditions
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)  # Contrast variations
    
    return image, label

def build_dataset(df, is_training=True):
    """Build tf.data pipeline"""
    ds = tf.data.Dataset.from_tensor_slices((df['path'].values, df[label_columns].values))
    
    if is_training:
        ds = ds.shuffle(buffer_size=len(df))
    
    ds = ds.map(parse_image, num_parallel_calls=AUTOTUNE)
    
    if is_training:
        # Apply skin-tone-specific augmentation (NO CutMix!)
        ds = ds.map(skin_tone_augment, num_parallel_calls=AUTOTUNE)
    
    ds = ds.batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
    return ds

train_dataset = build_dataset(train_df, is_training=True)
validation_dataset = build_dataset(validation_df, is_training=False)
test_dataset = build_dataset(test_df, is_training=False)

assert len(set(train_patients) & set(val_patients)) == 0
print("No overlap between train and validation patients")
assert len(set(train_patients) & set(test_patients)) == 0
print("No overlap between train and test patients")
assert len(set(val_patients) & set(test_patients)) == 0
print("No overlap between validation and test patients")

# ========================================================================
# SECTION 3B: FOCAL LOSS IMPLEMENTATION
# ========================================================================

def focal_loss(gamma=2.0, alpha=0.25):
    """
    Focal Loss for handling class imbalance.
    Focuses training on hard-to-classify examples.
    
    Args:
        gamma: Focusing parameter (higher = more focus on hard examples)
        alpha: Balancing parameter
    """
    def loss_fn(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        
        # Calculate cross entropy
        ce = -y_true * tf.math.log(y_pred)
        
        # Calculate focal weight
        focal_weight = tf.pow(1 - y_pred, gamma)
        
        # Apply focal loss
        focal_loss_val = alpha * focal_weight * ce
        
        return tf.reduce_sum(focal_loss_val, axis=-1)
    
    return loss_fn

print("\n--- Focal Loss configured ---")
print(f"  Gamma: {FOCAL_LOSS_GAMMA} (focus on hard examples)")
print(f"  Alpha: {FOCAL_LOSS_ALPHA} (balancing parameter)")




--- Building tf.data pipelines (IMAGE_SIZE=260)...
No overlap between train and validation patients
No overlap between train and test patients
No overlap between validation and test patients


In [ ]:
# ========================================================================
# SECTION 4: BUILDING THE ULTIMATE TRANSFER LEARNING MODEL
# ========================================================================

print(f"\n--- Building transfer learning model with EfficientNetV2B2...")

base_model = EfficientNetV2B2(
    include_top=False,
    weights='imagenet',
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)
base_model.trainable = False

inputs = Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(DROPOUT_RATE)(x)  # Increased dropout
outputs = Dense(len(label_columns), activation='softmax', dtype='float32')(x)

model = Model(inputs, outputs)
model.summary()



--- Building transfer learning model with EfficientNetV2B2...
27058176/35839040 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

KeyboardInterrupt: 

In [ ]:
# ========================================================================
# SECTION 5: STAGE 1 - TRAINING HEAD WITH FOCAL LOSS & CLASS WEIGHTS
# ========================================================================

print(f"\n--- STAGE 1: Training the classification head (with focal loss) ---")

model.compile(
    optimizer=AdamW(learning_rate=LEARNING_RATE_HEAD),
    loss=focal_loss(gamma=FOCAL_LOSS_GAMMA, alpha=FOCAL_LOSS_ALPHA),
    metrics=['accuracy']
)

checkpoint_head = ModelCheckpoint(
    'best_head_model_fitzpatrick.keras', 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max', 
    verbose=1
)

history_head = model.fit(
    train_dataset, 
    epochs=FEATURE_EXTRACTION_EPOCHS,
    validation_data=validation_dataset, 
    callbacks=[checkpoint_head],
    class_weight=class_weight_dict  # Apply class weights!
)

model.load_weights('best_head_model_fitzpatrick.keras')
print("✓ Best head model loaded")


In [ ]:
# ========================================================================
# SECTION 6: STAGE 2 - FINE-TUNING WITH LOWER LEARNING RATE
# ========================================================================

print(f"\n--- STAGE 2: Fine-tuning the entire model ---")

# Load best head weights
try:
    model.load_weights('best_head_model_fitzpatrick.keras')
    print("Successfully loaded best head model weights.")
except Exception as e:
    print(f"Error loading best head model weights: {e}")

base_model.trainable = True  # Unfreeze base model

# Recompile with lower learning rate
steps_per_epoch_ft = len(train_df) // BATCH_SIZE
lr_schedule_ft = CosineDecayRestarts(
    initial_learning_rate=LEARNING_RATE_FINE_TUNE_INITIAL,
    first_decay_steps=steps_per_epoch_ft * 10, 
    t_mul=2.0, 
    m_mul=0.9
)

optimizer_ft = AdamW(learning_rate=lr_schedule_ft, weight_decay=1e-5)

model.compile(
    optimizer=optimizer_ft,
    loss=focal_loss(gamma=FOCAL_LOSS_GAMMA, alpha=FOCAL_LOSS_ALPHA),
    metrics=['accuracy']
)

checkpoint_ft = ModelCheckpoint(
    'best_fine_tuned_model_fitzpatrick.keras', 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max', 
    verbose=1
)

early_stop_ft = EarlyStopping(
    monitor='val_loss', 
    patience=15,  # Increased patience for small dataset
    verbose=1
)

history_fine_tune = model.fit(
    train_dataset,
    epochs=FEATURE_EXTRACTION_EPOCHS + FINE_TUNING_EPOCHS,
    initial_epoch=FEATURE_EXTRACTION_EPOCHS,
    validation_data=validation_dataset,
    callbacks=[early_stop_ft, checkpoint_ft],
    class_weight=class_weight_dict  # Continue using class weights
)

model.load_weights('best_fine_tuned_model_fitzpatrick.keras')
print("✓ Best fine-tuned model loaded")


In [ ]:
# ========================================================================
# SECTION 7: ENHANCED EVALUATION WITH ADVANCED METRICS
# ========================================================================

from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
import seaborn as sns

def plot_history(history_head, history_fine_tune):
    """Plot training history"""
    acc = history_head.history['accuracy'] + history_fine_tune.history['accuracy']
    val_acc = history_head.history['val_accuracy'] + history_fine_tune.history['val_accuracy']
    loss = history_head.history['loss'] + history_fine_tune.history['loss']
    val_loss = history_head.history['val_loss'] + history_fine_tune.history['val_loss']
    epochs_range = range(len(acc))
    
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.axvline(x=len(history_head.epoch)-1, color='r', linestyle='--', label='Fine-tuning Start')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.axvline(x=len(history_head.epoch)-1, color='r', linestyle='--', label='Fine-tuning Start')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.show()

print("\n--- Plotting combined training history...")
plot_history(history_head, history_fine_tune)

def predict_with_tta(model, image_path, target_size):
    """Test-Time Augmentation with horizontal flip"""
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img)
    img_array_expanded = tf.expand_dims(img_array, axis=0)
    flipped_img_array = tf.image.flip_left_right(img_array_expanded)
    pred_original = model.predict(img_array_expanded, verbose=0)
    pred_flipped = model.predict(flipped_img_array, verbose=0)
    return np.mean([pred_original, pred_flipped], axis=0)[0]

def plot_precision_recall_curves(y_true_onehot, y_pred_probs, class_names):
    """Plot Precision-Recall curves for each class"""
    n_classes = len(class_names)
    
    # Calculate number of subplots needed
    n_cols = min(3, n_classes)
    n_rows = (n_classes + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    if n_classes == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for i, class_name in enumerate(class_names):
        precision, recall, _ = precision_recall_curve(y_true_onehot[:, i], y_pred_probs[:, i])
        
        axes[i].plot(recall, precision, linewidth=2)
        axes[i].set_xlabel('Recall', fontsize=10)
        axes[i].set_ylabel('Precision', fontsize=10)
        axes[i].set_title(f'Precision-Recall: {class_name}', fontsize=11, fontweight='bold')
        axes[i].grid(True, alpha=0.3)
        axes[i].set_xlim([0, 1])
        axes[i].set_ylim([0, 1])
        
        # Add baseline (random classifier)
        support = y_true_onehot[:, i].sum() / len(y_true_onehot)
        axes[i].axhline(y=support, color='r', linestyle='--', alpha=0.5, label=f'Baseline ({support:.2f})')
        axes[i].legend(loc='lower left', fontsize=9)
    
    # Hide unused subplots
    for i in range(n_classes, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

def show_classification_errors(df, predictions, true_labels_onehot, class_names, dataset_name, max_errors=12):
    """Display misclassified images with predicted vs true labels"""
    pred_classes = np.argmax(predictions, axis=1)
    true_classes = np.argmax(true_labels_onehot, axis=1)
    
    # Find all errors
    error_indices = np.where(pred_classes != true_classes)[0]
    
    if len(error_indices) == 0:
        print(f"\n🎉 No errors found in {dataset_name}!")
        return
    
    # Sample errors if too many
    if len(error_indices) > max_errors:
        error_indices = np.random.choice(error_indices, max_errors, replace=False)
    
    print(f"\n--- Error Analysis: {dataset_name} ---")
    print(f"Total errors: {len(np.where(pred_classes != true_classes)[0])} out of {len(df)}")
    
    # Calculate errors per class
    print("\nErrors by true class:")
    for i, class_name in enumerate(class_names):
        class_mask = true_classes == i
        class_errors = np.sum((pred_classes != true_classes) & class_mask)
        class_total = np.sum(class_mask)
        if class_total > 0:
            error_rate = class_errors / class_total * 100
            print(f"  {class_name:10s}: {class_errors:3d}/{class_total:3d} ({error_rate:5.1f}% error rate)")
    
    # Display sample errors
    n_cols = 4
    n_rows = (len(error_indices) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5*n_rows))
    if len(error_indices) == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    df_indexed = df.reset_index(drop=True)
    
    for idx, ax in enumerate(axes):
        if idx < len(error_indices):
            error_idx = error_indices[idx]
            img_path = df_indexed.loc[error_idx, 'path']
            
            true_class = class_names[true_classes[error_idx]]
            pred_class = class_names[pred_classes[error_idx]]
            confidence = predictions[error_idx, pred_classes[error_idx]] * 100
            
            # Load and display image
            img = load_img(img_path, target_size=(IMAGE_SIZE, IMAGE_SIZE))
            ax.imshow(img)
            ax.axis('off')
            
            # Add title with true vs predicted
            title = f"True: {true_class}\nPred: {pred_class} ({confidence:.1f}%)"
            ax.set_title(title, fontsize=9, color='red', fontweight='bold')
        else:
            ax.axis('off')
    
    plt.tight_layout()
    plt.suptitle(f'Sample Misclassifications - {dataset_name}', fontsize=14, fontweight='bold', y=1.00)
    plt.show()

def class_wise_accuracy_by_fitzpatrick(df, predictions, true_labels_onehot, class_names):
    """Cross-tabulation: predicted class accuracy by original Fitzpatrick type"""
    pred_classes = np.argmax(predictions, axis=1)
    true_classes = np.argmax(true_labels_onehot, axis=1)
    
    # Get original Fitzpatrick types (before grouping)
    fitzpatrick_types = df['fitzpatrick_skin_type'].values
    
    print("\n--- Class-wise Accuracy by Original Fitzpatrick Type ---")
    
    # Create cross-tabulation
    fitzpatrick_unique = sorted(df['fitzpatrick_skin_type'].unique())
    
    # Build accuracy table
    results = []
    for fitz_type in fitzpatrick_unique:
        fitz_mask = fitzpatrick_types == fitz_type
        fitz_count = np.sum(fitz_mask)
        
        if fitz_count == 0:
            continue
        
        row = {'Fitzpatrick': fitz_type, 'Total': fitz_count}
        
        # Calculate accuracy for each predicted class
        for i, class_name in enumerate(class_names):
            class_correct = np.sum((pred_classes == i) & (true_classes == i) & fitz_mask)
            class_total = np.sum((true_classes == i) & fitz_mask)
            
            if class_total > 0:
                accuracy = class_correct / class_total * 100
                row[f'{class_name}'] = f"{class_correct}/{class_total} ({accuracy:.1f}%)"
            else:
                row[f'{class_name}'] = "N/A"
        
        # Overall accuracy for this Fitzpatrick type
        overall_correct = np.sum((pred_classes == true_classes) & fitz_mask)
        overall_acc = overall_correct / fitz_count * 100
        row['Overall'] = f"{overall_correct}/{fitz_count} ({overall_acc:.1f}%)"
        
        results.append(row)
    
    # Create DataFrame and display
    results_df = pd.DataFrame(results)
    print("\n" + results_df.to_string(index=False))
    
    # Visualize as heatmap
    heatmap_data = []
    heatmap_labels = []
    
    for fitz_type in fitzpatrick_unique:
        fitz_mask = fitzpatrick_types == fitz_type
        fitz_count = np.sum(fitz_mask)
        
        if fitz_count == 0:
            continue
        
        accuracies = []
        for i in range(len(class_names)):
            class_correct = np.sum((pred_classes == i) & (true_classes == i) & fitz_mask)
            class_total = np.sum((true_classes == i) & fitz_mask)
            
            if class_total > 0:
                accuracies.append(class_correct / class_total * 100)
            else:
                accuracies.append(0)
        
        heatmap_data.append(accuracies)
        heatmap_labels.append(fitz_type)
    
    if heatmap_data:
        plt.figure(figsize=(8, 6))
        sns.heatmap(
            heatmap_data, 
            annot=True, 
            fmt='.1f', 
            cmap='RdYlGn',
            xticklabels=class_names,
            yticklabels=heatmap_labels,
            vmin=0,
            vmax=100,
            cbar_kws={'label': 'Accuracy (%)'}
        )
        plt.title('Accuracy by Fitzpatrick Type and Predicted Class')
        plt.xlabel('Predicted Class')
        plt.ylabel('Original Fitzpatrick Type')
        plt.tight_layout()
        plt.show()

def evaluate_with_metrics(model, df, dataset_name, patient_list):
    """Comprehensive evaluation with all metrics"""
    print(f"\n{'='*60}")
    print(f"EVALUATING: {dataset_name}")
    print(f"{'='*60}")
    
    # Collect predictions
    predictions = []
    true_labels_onehot = df[label_columns].values
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} TTA"):
        tta_pred = predict_with_tta(model, row['path'], (IMAGE_SIZE, IMAGE_SIZE))
        predictions.append(tta_pred)
    
    predictions = np.array(predictions)
    pred_classes = np.argmax(predictions, axis=1)
    true_classes = np.argmax(true_labels_onehot, axis=1)
    
    # Overall accuracy
    accuracy = np.mean(pred_classes == true_classes)
    
    # 1. Classification report
    print(f"\n1. CLASSIFICATION REPORT:")
    print(classification_report(
        true_classes, 
        pred_classes, 
        target_names=label_columns,
        digits=3
    ))
    
    # 2. Confusion matrix
    print(f"\n2. CONFUSION MATRIX:")
    cm = confusion_matrix(true_classes, pred_classes)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=label_columns, 
                yticklabels=label_columns)
    plt.title(f'Confusion Matrix - {dataset_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    
    # 3. Precision-Recall curves
    print(f"\n3. PRECISION-RECALL CURVES:")
    plot_precision_recall_curves(true_labels_onehot, predictions, label_columns)
    
    # 4. Error analysis
    print(f"\n4. ERROR ANALYSIS:")
    show_classification_errors(df, predictions, true_labels_onehot, label_columns, dataset_name)
    
    # 5. Class-wise accuracy by Fitzpatrick type
    print(f"\n5. ACCURACY BY ORIGINAL FITZPATRICK TYPE:")
    class_wise_accuracy_by_fitzpatrick(df, predictions, true_labels_onehot, label_columns)
    
    # Summary
    print(f"\n{dataset_name} SUMMARY:")
    print(f"  Patients: {len(patient_list)}")
    print(f"  Images: {len(df)}")
    print(f"  Overall Accuracy: {accuracy:.4f}")
    
    return accuracy

# Evaluate on validation set
val_accuracy = evaluate_with_metrics(
    model, 
    validation_df, 
    "VALIDATION SET", 
    val_patients
)

# Evaluate on test set
test_accuracy = evaluate_with_metrics(
    model, 
    test_df, 
    "TEST SET", 
    test_patients
)

# Final summary
best_val_acc_training = max(history_fine_tune.history.get('val_accuracy', [0]))

print(f"\n{'='*60}")
print(f"FINAL RESULTS - {('3-WAY' if USE_3WAY_CLASSIFICATION else '6-WAY')} CLASSIFICATION")
print(f"{'='*60}")
print(f"Best Validation Accuracy (during training): {best_val_acc_training:.4f}")
print(f"\nTest-Time Augmentation Results:")
print(f"  Validation Set TTA Accuracy: {val_accuracy:.4f}")
print(f"  Test Set TTA Accuracy:       {test_accuracy:.4f}")
print(f"{'='*60}")
